7.1 Создание составных типов из примитивных

In [1]:
typeof(42)

Int64


In [2]:
typeof('A')

Char


In [3]:
typeof("hello")

String


In [4]:
isprimitivetype(Int8)

true


In [5]:
isprimitivetype(Char)

true


In [6]:
isprimitivetype(String)

false


In [7]:
struct Archer
    name::String
    health::Int
    arrows::Int
end

In [8]:
robin = Archer("Robin Hood", 30, 24)

Archer("Robin Hood", 30, 24)


In [9]:
robin.name

"Robin Hood"


In [10]:
robin.health

30


In [11]:
robin.arrows

24


In [12]:
robin = Dict(
    "name" => "Robin Hood",
    "health" => 30,
    "arrows" => 24
)

Dict{String, Any} with 3 entries:
  "name" => "Robin Hood"
  "health" => 30
  "arrows" => 24


In [13]:
robin["name"]

"Robin Hood"


In [14]:
robin["arrows"]

24


7.2 Обследование иерархий типов

In [15]:
supertype(Int64)

Signed


In [16]:
supertype(Signed)

Integer


In [17]:
supertype(Integer)

Real


In [18]:
supertype(Real)

Number


In [19]:
supertype(Number)

Any


In [20]:
supertype(Any)

Any


In [21]:
function findroot(T)
    T2 = supertype(T)
    println(T)
    if T2 != T
        findroot(T2)
    end
end

findroot (generic function with 1 method)


In [22]:
findroot(typeof(42))

Int64
Signed
Integer
Real
Number
Any


In [23]:
supertype(Any)

Any


In [24]:
anything = Any[42, 8]

2-element Vector{Any}:
 42
  8


In [25]:
subtypes(Integer)

3-element Vector{Any}:
 Bool
 Signed
 Unsigned


In [26]:
subtypes(Signed)

6-element Vector{Any}:
 BigInt
 Int128
 Int16
 Int32
 Int64
 Int8


In [27]:
subtypes(Unsigned)

5-element Vector{Any}:
 UInt128
 UInt16
 UInt32
 UInt64
 UInt8


In [28]:
String <: Any

true


In [29]:
String <: Integer

false


In [30]:
Int8 <: Integer

true


In [31]:
Float64 <: Integer

false


In [32]:
Int8 <: Number

true


In [33]:
Float64 <: Number

true


7.3 Создание симулятора сражений

7.3.1 Определение типов воинов

In [34]:
abstract type Warrior end

In [35]:
mutable struct Archer <: Warrior
    name::String
    health::Int
    arrows::Int
end

In [36]:
mutable struct Pikeman <: Warrior
    name::String
    health::Int
end

In [37]:
mutable struct Knight <: Warrior
    name::String
    health::Int
    mounted::Bool
end

In [38]:
subtypes(Warrior)

3-element Vector{Any}:
 Archer
 Knight
 Pikeman


In [39]:
Archer <: Warrior

true


In [40]:
Knight <: Warrior

true


In [41]:
Pikeman <: Warrior

true


In [42]:
robin = Archer("Robin Hood", 34, 24)

Archer("Robin Hood", 34, 24)


In [43]:
white = Knight("Lancelot", 34, true)

Knight("Lancelot", 34, true)


In [44]:
black = Knight("Morien", 35, false)

Knight("Morien", 35, false)


In [45]:
boris = Pikeman("Boris", 30)

Pikeman("Boris", 30)


7.3.2 Добавление воинам поведения

In [46]:
function shoot!(archer::Archer)
    if archer.arrows > 0
        archer.arrows -= 1
    end
    archer
end

shoot! (generic function with 1 method)


In [47]:
function resupply!(archer::Archer)
    archer.arrows = 24
    archer
end

resupply! (generic function with 1 method)


In [48]:
function mount!(knight::Knight)
    knight.mounted = true
    knight
end

mount! (generic function with 1 method)


In [49]:
function dismount!(knight::Knight)
    knight.mounted = false
    knight
end

dismount! (generic function with 1 method)


In [50]:
shoot!(robin)

Archer("Robin Hood", 34, 23)


In [51]:
shoot!(robin)

Archer("Robin Hood", 34, 22)


In [52]:
shoot!(robin)

Archer("Robin Hood", 34, 21)


In [53]:
resupply!(robin)

Archer("Robin Hood", 34, 24)


In [54]:
dismount!(white)

Knight("Lancelot", 34, false)


In [55]:
mount!(white)

Knight("Lancelot", 34, true)


7.3.3 Использование множественной диспетчеризации для вызова методов

In [56]:
function attack!(a::Archer, b::Archer)
    if a.arrows > 0
        shoot!(a)
        damage = 6 + rand(1:6)
        b.health = max(b.health - damage, 0)
    end
    a.health, b.health
end

attack! (generic function with 1 method)


In [57]:
function attack!(a::Archer, b::Knight)
    if a.arrows > 0
        shoot!(a)
        damage = rand(1:6)
        if b.mounted
            damage += 3
        end
        b.health = max(b.health - damage, 0)
    end
    a.health, b.health
end

attack! (generic function with 2 methods)


In [58]:
function attack!(a::Knight, b::Knight)
    a.health = max(a.health - rand(1:6), 0)
    b.health = max(b.health - rand(1:6), 0)
    a.health, b.health
end

attack! (generic function with 3 methods)


In [59]:
function attack!(a::Archer, b::Pikeman)
    if a.arrows > 0
        shoot!(a)
        damage = 4 + rand(1:6)
        b.health = max(b.health - damage, 0)
    end
    a.health, b.health
end

attack! (generic function with 4 methods)


In [60]:
methods(attack!)

# 4 methods for generic function "attack!" from Main:
 [1] attack!(a::Archer, b::Pikeman)
     @ none:2
 [2] attack!(a::Knight, b::Knight)
     @ none:2
 [3] attack!(a::Archer, b::Knight)
     @ none:2
 [4] attack!(a::Archer, b::Archer)
     @ none:2


In [61]:
attack!(robin, white)

(34, 30)


In [62]:
attack!(black, white)

(32, 26)


In [63]:
attack!(robin, boris)

(34, 24)


7.4 Как Julia выбирает метод, который следует вызвать

In [64]:
methods(attack!)

# 4 methods for generic function "attack!" from Main:
 [1] attack!(a::Archer, b::Pikeman)
     @ none:2
 [2] attack!(a::Knight, b::Knight)
     @ none:2
 [3] attack!(a::Archer, b::Knight)
     @ none:2
 [4] attack!(a::Archer, b::Archer)
     @ none:2


In [65]:
(typeof(robin), typeof(white))

(Archer, Knight)


7.4.1 Сравнение множественной диспетчеризации Julia с объектно ориентированными языками

In [66]:
# Демонстрация принципа одиночной диспетчеризации (ООП-стиль через ветвление)
function single_dispatch_attack(archer::Archer, opponent)
    if typeof(opponent) == Archer
        "Лучник атакует Лучника"
    elseif typeof(opponent) == Knight
        "Лучник атакует Рыцаря"
    elseif typeof(opponent) == Pikeman
        "Лучник атакует Копейщика"
    end
end

single_dispatch_attack (generic function with 1 method)


In [67]:
single_dispatch_attack(robin, white)

"Лучник атакует Рыцаря"


In [68]:
single_dispatch_attack(robin, boris)

"Лучник атакует Копейщика"


7.4.2 Чем множественная диспетчеризация отличается от перегрузки функций?

In [69]:
function battle!(a::Warrior, b::Warrior)
    attack!(a, b)
    if a.health == 0 && b.health == 0
        println(a.name, " and ", b.name, " destroyed each other")
    elseif a.health == 0
        println(b.name, " defeated ", a.name)
    elseif b.health == 0
        println(a.name, " defeated ", b.name)
    else
        println(b.name, " survived attack from ", a.name)
    end
end

battle! (generic function with 1 method)


In [70]:
battle!(robin, white)

Lancelot survived attack from Robin Hood


In [71]:
battle!(robin, white)

Lancelot survived attack from Robin Hood


In [72]:
battle!(robin, white)

Lancelot survived attack from Robin Hood


In [73]:
white.health

1


In [74]:
robin.health

34


In [75]:
robin.arrows

19
